# Structured Fusion Search (audios4-CV, prior-adjusted)

**Goal:** find the minimum-model fusion that maximises recall at high-precision operating points,
using an explicit audio x text pairing protocol instead of exhaustive combinatorics.

**Base models (10):**
- **Audio (4):** `whisper_rf`, `whisper_xgb`, `wavlm_xgb`, `wavlm_rf`
- **Text  (6):** `text_top10_xgb`, `text_top10_rf`, `text_top15_xgb`, `text_top15_rf`,
  `text_stylo_xgb`, `text_stylo_rf`

**Search phases (all on audios4-CV, audios5 held out end-to-end):**
1. **Phase 1 - 2-way pair.** Every (audio x text) pair (4 x 6 = 24), alpha sweep on weighted average, pick best by cv_f1.
2. **Phase 2 - Add a 3rd.** From the Phase-1 winner, try adding every remaining base model; report the cv_f1 delta. A 3rd is "worth it" only if it adds >= ADD_EPS (default 0.01).
3. **Phase 3 - Add a 4th.** Same logic from the 3-way winner.

**Protocol guarantees:**
- `audios2` always in train; 5-fold StratifiedKFold on `audios4`; threshold picked globally on the concatenated OOF vector.
- `audios5` touched **once** at the very end for a one-shot test at the frozen cv_thr -> reports gap_f1.
- All base classifiers use `SPW_DEPLOY = (1-0.17)/0.17 ~= 4.88` so CV mimics the audios5 class prior.
- Metrics columns: `cv_f1, cv_prec, cv_rec, cv_thr, fold_f1_mean, fold_f1_std, test_f1_at_cv, test_prec_at_cv, test_rec_at_cv, gap_f1, cv_rec@P{60..95}` in 5-pp steps.


In [ ]:
# ================================================================
# CONFIGURATION
# ================================================================
from pathlib import Path

TRAIN_FOLDERS = ['audios2', 'audios4']
CV_TARGET     = 'audios4'
TEST_FOLDER   = 'audios5'

DEPLOY_POS_RATE = 0.17
SPW_DEPLOY      = (1.0 - DEPLOY_POS_RATE) / DEPLOY_POS_RATE   # ~4.88

PREC_TARGETS = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
RANDOM_SEED  = 42
CV_FOLDS     = 5

# Minimum cv_f1 gain to accept a 3rd or 4th model
ADD_EPS = 0.01

NB_DIR   = Path('.').resolve()
SAVE_DIR = NB_DIR / 'checkpoints_fusion_v2'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Cache file conventions (same as fusion_text_wavlm.ipynb; auto-fallbacks below)
TEXT_CACHE    = lambda n: NB_DIR / f'{n}_features.csv'
WAVLM_CACHE   = lambda n: NB_DIR / f'{n}_whole_pretrained.csv'
WHISPER_CACHE = lambda n: NB_DIR / f'{n}_whisper_whole.csv'
GT_CACHE      = lambda n: NB_DIR / f'{n}GT.csv'

LABEL_MAP = {
    'read':1,'cheating':1,'reading':1,'scripted':1,'yes':1,'1':1,1:1,
    'spontaneous':0,'not cheating':0,'not_cheating':0,'no':0,'0':0,0:0,'genuine':0,
}

print(f'Train: {TRAIN_FOLDERS}  |  CV: {CV_TARGET}  |  Test: {TEST_FOLDER}')
print(f'Deploy prior: {DEPLOY_POS_RATE:.0%}  ->  SPW_DEPLOY = {SPW_DEPLOY:.4f}')
print(f'Save -> {SAVE_DIR}')


In [ ]:
import json, itertools, warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              confusion_matrix)
warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)

# Text feature groups (must match text_cheating_detection.ipynb)
GROUPS = {
    'disfluency':  ['filler_rate','filler_count','repetition_rate','repair_rate',
                    'discourse_marker_rate','hedge_rate'],
    'stylometric': ['ttr','mattr','mtld','complex_word_rate','avg_word_length',
                    'n_words','n_unique_words','avg_sentence_length','std_sentence_length',
                    'fragment_rate','n_sentences','self_ref_rate',
                    'noun_rate','verb_rate','adj_rate'],
    'pause':       ['pause_mean','pause_std','pause_median','pause_skew','long_pause_rate',
                    'pause_ratio','n_pauses','pause_regularity',
                    'pause_before_content_ratio','pause_before_function_ratio',
                    'mid_phrase_pause_rate','words_per_sec','articulation_rate',
                    'initial_pause','longest_pause'],
    'suspicious':  ['suspicious_gap_count','suspicious_gap_ratio'],
    'formal_ai':   ['formal_transition_count','formal_transition_rate',
                    'ai_phrase_count','ai_phrase_rate'],
    'prosodic':    ['f0_mean','f0_std','f0_range','f0_skew','f0_slope',
                    'energy_mean','energy_std','speaking_rate_std'],
    'voice_q':     ['jitter_local','shimmer_local','hnr_mean'],
    'perplexity':  ['mean_perplexity','burstiness'],
}
ALL_TEXT_FEATS = [f for g in GROUPS.values() for f in g]
STYLO_FEATS    = GROUPS['stylometric']
print(f'Text feature universe: {len(ALL_TEXT_FEATS)}')
print(f'Stylometric features : {len(STYLO_FEATS)}')


## 1. Load batches (text + WavLM whole-pool + Whisper)

In [ ]:
def load_gt(name):
    gt = pd.read_csv(GT_CACHE(name))
    fn_col  = next(c for c in gt.columns if c.lower() in ('filename','file','name'))
    lbl_col = next(c for c in gt.columns if c.lower() in ('label','class','cheating','gt','label_int','ground_truth'))
    gt = gt.rename(columns={fn_col:'filename', lbl_col:'label_raw'})
    gt['label_int'] = gt['label_raw'].map(
        lambda x: LABEL_MAP.get(x, LABEL_MAP.get(str(x).lower().strip(), -1)))
    return gt[gt['label_int'].isin([0,1])][['filename','label_int']]

def _wavlm_path(n):
    p = WAVLM_CACHE(n)
    if p.exists(): return p
    alt = NB_DIR / f'{n}_wavlm_whole.csv'
    if alt.exists(): return alt
    raise FileNotFoundError(f'{p} or {alt}')

def load_batch(name):
    gt = load_gt(name)
    text    = pd.read_csv(TEXT_CACHE(name))
    wavlm   = pd.read_csv(_wavlm_path(name))
    whisper = pd.read_csv(WHISPER_CACHE(name))
    df = (gt.merge(text,    on='filename', how='inner')
            .merge(wavlm,   on='filename', how='inner')
            .merge(whisper, on='filename', how='inner'))
    df['batch'] = name
    return df

batches = {b: load_batch(b) for b in TRAIN_FOLDERS + [TEST_FOLDER]}

_first = batches[TRAIN_FOLDERS[0]]
wavlm_cols   = [c for c in _first.columns if c.startswith('wavlm_')
                and not c.startswith('wavlm_mean_') and not c.startswith('wavlm_std_')]
whisper_cols = [c for c in _first.columns if c.startswith('whisper_')]
text_cols    = [c for c in ALL_TEXT_FEATS if c in _first.columns]
stylo_cols   = [c for c in STYLO_FEATS if c in _first.columns]

print(f'WavLM dims: {len(wavlm_cols)}  |  Whisper dims: {len(whisper_cols)}')
print(f'Text feats: {len(text_cols)}  |  Stylo feats: {len(stylo_cols)}')
for name, df in batches.items():
    y = df['label_int'].values
    print(f'  {name}: n={len(df):4d}  cheat={int((y==1).sum()):3d}  honest={int((y==0).sum()):3d}')


## 2. Rank text features (for top-10 / top-15 lists)

Importance computed from an XGBoost trained on the **always-train** batch (audios2) only - prevents
any leakage of audios4 label information into the feature-selection step. The selected lists are
then frozen and used across all CV folds.

In [ ]:
def _xgb_for(n_feats):
    colsample = 0.3 if n_feats > 500 else 0.8
    return xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=colsample, min_child_weight=3,
        scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
        random_state=RANDOM_SEED)

_rank_src = [b for b in TRAIN_FOLDERS if b != CV_TARGET]
_df_rank  = pd.concat([batches[b] for b in _rank_src], ignore_index=True)
X_rank = _df_rank[text_cols].fillna(0).values
y_rank = _df_rank['label_int'].values

_sc  = StandardScaler().fit(X_rank)
_rkr = _xgb_for(len(text_cols))
_rkr.fit(_sc.transform(X_rank), y_rank)
_imp = pd.Series(_rkr.feature_importances_, index=text_cols).sort_values(ascending=False)

TOP10_FEATS = _imp.head(10).index.tolist()
TOP15_FEATS = _imp.head(15).index.tolist()

print(f'Ranking source: {_rank_src}  (n={len(_df_rank)})')
print(f'\nTop-10 text features:')
for f in TOP10_FEATS: print(f'  {f:30s}  imp={_imp[f]:.4f}')
print(f'\nTop-15 text features (positions 11-15):')
for f in TOP15_FEATS[10:]: print(f'  {f:30s}  imp={_imp[f]:.4f}')
print(f'\nStylometric features ({len(stylo_cols)}):')
print(f'  {stylo_cols}')


## 3. Metrics helpers + rec@P{60..95}

In [ ]:
def best_f1_on(proba, y, thr_grid=np.arange(0.20, 0.81, 0.01)):
    best_f1, best_thr = -1.0, 0.5
    for thr in thr_grid:
        f = f1_score(y, (proba >= thr).astype(int), zero_division=0)
        if f > best_f1: best_f1, best_thr = f, thr
    return float(best_thr), float(best_f1)

def metrics_at(proba, y, thr):
    pred = (proba >= thr).astype(int)
    cm = confusion_matrix(y, pred, labels=[0,1])
    return dict(
        thr=round(float(thr), 3),
        precision=round(precision_score(y, pred, zero_division=0), 4),
        recall   =round(recall_score   (y, pred, zero_division=0), 4),
        f1       =round(f1_score       (y, pred, zero_division=0), 4),
        tp=int(cm[1,1]), fp=int(cm[0,1]), fn=int(cm[1,0]), tn=int(cm[0,0]),
    )

def rec_at_prec(proba, y, targets=PREC_TARGETS, min_tp=3):
    out = {}
    for t in targets:
        best_rec, best_thr = None, None
        for thr in np.arange(0.99, 0.10, -0.01):
            pred = (proba >= thr).astype(int)
            cm = confusion_matrix(y, pred, labels=[0,1])
            if cm[1,1] < min_tp: continue
            p = precision_score(y, pred, zero_division=0)
            r = recall_score   (y, pred, zero_division=0)
            if p >= t and (best_rec is None or r > best_rec):
                best_rec, best_thr = r, thr
        key = f'rec@P{int(round(t*100))}'
        out[key] = round(best_rec, 4) if best_rec is not None else 0.0
    return out

def per_fold_stats(proba, y, fold_assign, global_thr):
    rows = []
    for fi in sorted(set(fold_assign.tolist())):
        mk = fold_assign == fi
        y_f, p_f = y[mk], proba[mk]
        pred_f = (p_f >= global_thr).astype(int)
        rows.append(dict(
            fold=int(fi), n=int(mk.sum()), n_cheat=int((y_f==1).sum()),
            f1_at_global=round(f1_score(y_f, pred_f, zero_division=0), 4),
            own_best_thr=round(best_f1_on(p_f, y_f)[0], 3),
        ))
    pf = pd.DataFrame(rows)
    return pf, dict(
        fold_f1_mean=round(float(pf['f1_at_global'].mean()), 4),
        fold_f1_std =round(float(pf['f1_at_global'].std()),  4),
        fold_thr_mean=round(float(pf['own_best_thr'].mean()), 3),
        fold_thr_std =round(float(pf['own_best_thr'].std()),  3),
    )


## 4. Base-model registry (10 heads)

In [ ]:
def make_rf():
    return RandomForestClassifier(
        n_estimators=500, max_depth=8, min_samples_leaf=3,
        class_weight={0: 1.0, 1: float(SPW_DEPLOY)},
        n_jobs=-1, random_state=RANDOM_SEED)

# Feature extractors
def X_wavlm(df):   return df[wavlm_cols].fillna(0).values
def X_whisper(df): return df[whisper_cols].fillna(0).values
def X_top10(df):   return df[TOP10_FEATS].fillna(0).values
def X_top15(df):   return df[TOP15_FEATS].fillna(0).values
def X_stylo(df):   return df[stylo_cols].fillna(0).values

# Registry: tag -> (X_fn, make_clf, family)
BASE_REGISTRY = {
    # audio
    'whisper_rf':     (X_whisper, make_rf,                              'audio'),
    'whisper_xgb':    (X_whisper, lambda: _xgb_for(len(whisper_cols)),  'audio'),
    'wavlm_xgb':      (X_wavlm,   lambda: _xgb_for(len(wavlm_cols)),    'audio'),
    'wavlm_rf':       (X_wavlm,   make_rf,                              'audio'),
    # text
    'text_top10_xgb': (X_top10,   lambda: _xgb_for(10),                 'text'),
    'text_top10_rf':  (X_top10,   make_rf,                              'text'),
    'text_top15_xgb': (X_top15,   lambda: _xgb_for(15),                 'text'),
    'text_top15_rf':  (X_top15,   make_rf,                              'text'),
    'text_stylo_xgb': (X_stylo,   lambda: _xgb_for(len(stylo_cols)),    'text'),
    'text_stylo_rf':  (X_stylo,   make_rf,                              'text'),
}

AUDIO_BASES = [m for m, (_,_,fam) in BASE_REGISTRY.items() if fam == 'audio']
TEXT_BASES  = [m for m, (_,_,fam) in BASE_REGISTRY.items() if fam == 'text']
print(f'Audio bases ({len(AUDIO_BASES)}): {AUDIO_BASES}')
print(f'Text bases  ({len(TEXT_BASES)}): {TEXT_BASES}')

def fit_and_score(X_fn, make_clf, df_tr, df_te):
    Xt, yt = X_fn(df_tr), df_tr['label_int'].values
    Xe     = X_fn(df_te)
    sc = StandardScaler().fit(Xt)
    m  = make_clf()
    m.fit(sc.transform(Xt), yt)
    return m.predict_proba(sc.transform(Xe))[:, 1]


## 5. audios4-CV OOF probas for all 10 bases

audios2 concatenated to every fold's train side. 5-fold stratified on audios4.

In [ ]:
always_train_names = [b for b in TRAIN_FOLDERS if b != CV_TARGET]
df_always_tr = (pd.concat([batches[b] for b in always_train_names], ignore_index=True)
                if always_train_names else None)
df_cv_target = batches[CV_TARGET].reset_index(drop=True)
y_cv  = df_cv_target['label_int'].values
fn_cv = df_cv_target['filename'].values

print(f'audios4-CV: target={CV_TARGET}  n={len(df_cv_target)}  '
      f'cheat={int((y_cv==1).sum())}  honest={int((y_cv==0).sum())}')
if df_always_tr is not None:
    print(f'Always train: {always_train_names}  n={len(df_always_tr)}  '
          f'cheat={int((df_always_tr["label_int"]==1).sum())}')

cv_scores   = {m: np.full(len(df_cv_target), np.nan) for m in BASE_REGISTRY}
fold_assign = np.full(len(df_cv_target), -1, dtype=int)

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
for fi, (tr_idx, va_idx) in enumerate(skf.split(df_cv_target, y_cv)):
    df_tr_fold = df_cv_target.iloc[tr_idx]
    df_va_fold = df_cv_target.iloc[va_idx]
    df_tr = (pd.concat([df_always_tr, df_tr_fold], ignore_index=True)
             if df_always_tr is not None else df_tr_fold)
    fold_assign[va_idx] = fi
    print(f'  fold {fi+1}/{CV_FOLDS}  n_train={len(df_tr)}  '
          f'n_val={len(df_va_fold)}  val_cheat={int(df_va_fold["label_int"].sum())}')
    for tag, (X_fn, make_clf, _) in BASE_REGISTRY.items():
        p = fit_and_score(X_fn, make_clf, df_tr, df_va_fold)
        cv_scores[tag][va_idx] = p

assert not any(np.isnan(v).any() for v in cv_scores.values()), 'CV OOF has NaNs'
print(f'\nCV OOF rows: {len(y_cv)}  cheat={int((y_cv==1).sum())}  honest={int((y_cv==0).sum())}')


## 6. Base-model CV metrics + per-fold stats

In [ ]:
base_rows  = []
base_folds = []
for tag in BASE_REGISTRY:
    p = cv_scores[tag]
    thr, _ = best_f1_on(p, y_cv)
    m  = metrics_at(p, y_cv, thr)
    rp = rec_at_prec(p, y_cv)
    pf, agg = per_fold_stats(p, y_cv, fold_assign, thr)
    pf.insert(0, 'base', tag)
    base_folds.append(pf)
    base_rows.append({'base': tag,
                      'family': BASE_REGISTRY[tag][2],
                      **m, **rp, **agg})

base_df = pd.DataFrame(base_rows).sort_values('f1', ascending=False).reset_index(drop=True)
base_folds_df = pd.concat(base_folds, ignore_index=True)

show = ['base','family','thr','f1','precision','recall',
        'fold_f1_mean','fold_f1_std'] + [f'rec@P{int(t*100)}' for t in PREC_TARGETS]
print('Base-model audios4-CV metrics (sorted by CV F1):')
print(base_df[show].to_string(index=False))


## 7. Phase 1 - all (audio x text) pairs, alpha sweep

In [ ]:
def grid_2way(step=0.05):
    return [(round(w, 2), round(1-w, 2)) for w in np.arange(0.0, 1.001, step)]

def score_combo(members, weights, cv_vec, y):
    thr, _ = best_f1_on(cv_vec, y)
    m  = metrics_at(cv_vec, y, thr)
    rp = rec_at_prec(cv_vec, y)
    pf, agg = per_fold_stats(cv_vec, y, fold_assign, thr)
    return {'members': tuple(members), 'weights': tuple(weights),
            **m, **rp, **agg, '_proba_cv': cv_vec, '_pf': pf}

pair_records = []
for a in AUDIO_BASES:
    for t in TEXT_BASES:
        best = None
        for wa, wt in grid_2way(step=0.05):
            vec = wa * cv_scores[a] + wt * cv_scores[t]
            _, f1 = best_f1_on(vec, y_cv)
            if best is None or f1 > best[0]:
                best = (f1, wa, wt, vec)
        _, wa, wt, vec = best
        rec = score_combo([a, t], [wa, wt], vec, y_cv)
        rec['tag']   = f'pair:{a}+{t}'
        rec['audio'] = a
        rec['text']  = t
        pair_records.append(rec)

pairs_df = (pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')}
                          for r in pair_records])
            .sort_values('f1', ascending=False).reset_index(drop=True))

show = ['tag','weights','thr','f1','precision','recall',
        'fold_f1_mean','fold_f1_std'] + [f'rec@P{int(t*100)}' for t in PREC_TARGETS]
print(f'All (audio x text) pairs = {len(pairs_df)}. Top 10 by CV F1:')
print(pairs_df[show].head(10).to_string(index=False))
print('\nBest by rec@P85:')
print(pairs_df.sort_values('rec@P85', ascending=False).head(5)[show].to_string(index=False))
print('\nBest by rec@P90:')
print(pairs_df.sort_values('rec@P90', ascending=False).head(5)[show].to_string(index=False))

best_pair = sorted(pair_records, key=lambda r: -r['f1'])[0]
print(f'\nPhase 1 winner (by CV F1): {best_pair["tag"]}')
print(f'  weights = {best_pair["weights"]}  cv_f1 = {best_pair["f1"]:.4f}')


## 8. Phase 2 - add a 3rd model to the Phase-1 winner

Grid over 3-way simplex (step 0.1). Reports cv_f1 delta vs the 2-way winner.

In [ ]:
def grid_3way(step=0.1):
    out = []
    for w1 in np.arange(0, 1.001, step):
        for w2 in np.arange(0, 1.001 - w1 + 1e-9, step):
            w3 = 1.0 - w1 - w2
            if w3 < -1e-9: continue
            out.append((round(w1, 2), round(w2, 2), round(max(0, w3), 2)))
    return out

pair_members = list(best_pair['members'])
pair_f1_ref  = best_pair['f1']

triple_records = []
for add_model in BASE_REGISTRY:
    if add_model in pair_members: continue
    trio = pair_members + [add_model]
    best = None
    for w in grid_3way(step=0.1):
        vec = sum(wi * cv_scores[m] for wi, m in zip(w, trio))
        _, f1 = best_f1_on(vec, y_cv)
        if best is None or f1 > best[0]:
            best = (f1, w, vec)
    _, w, vec = best
    rec = score_combo(trio, w, vec, y_cv)
    rec['tag']      = f'3way:{"+".join(trio)}'
    rec['added']    = add_model
    rec['delta_f1'] = round(rec['f1'] - pair_f1_ref, 4)
    triple_records.append(rec)

triples_df = (pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')}
                            for r in triple_records])
              .sort_values('f1', ascending=False).reset_index(drop=True))

show = ['tag','weights','thr','f1','delta_f1','precision','recall',
        'fold_f1_mean','fold_f1_std'] + [f'rec@P{int(t*100)}' for t in PREC_TARGETS]
print(f'3-way candidates (Phase 1 winner + each remaining base):')
print(triples_df[show].to_string(index=False))

accepted_3way = [r for r in triple_records if r['delta_f1'] >= ADD_EPS]
if accepted_3way:
    best_triple = sorted(accepted_3way, key=lambda r: -r['f1'])[0]
    print(f'\nAccepted 3-way winner (delta_f1 >= {ADD_EPS:.2f}): {best_triple["tag"]}')
    print(f'  weights = {best_triple["weights"]}  cv_f1 = {best_triple["f1"]:.4f}  delta = {best_triple["delta_f1"]:+.4f}')
else:
    best_triple = None
    print(f'\nNo 3-way beats the 2-way by >= {ADD_EPS:.2f} on CV F1. Sticking with 2-way.')


## 9. Phase 3 - add a 4th (only if a 3-way was accepted)

In [ ]:
def grid_4way(step=0.1):
    out = []
    s = np.arange(0, 1.001, step)
    for w1 in s:
        for w2 in s:
            if w1 + w2 > 1.001: continue
            for w3 in s:
                if w1 + w2 + w3 > 1.001: continue
                w4 = 1.0 - w1 - w2 - w3
                if w4 < -1e-9: continue
                out.append((round(w1, 2), round(w2, 2), round(w3, 2), round(max(0, w4), 2)))
    return out

quadruple_records = []
best_quadruple    = None
quads_df          = pd.DataFrame()

if best_triple is not None:
    triple_members = list(best_triple['members'])
    triple_f1_ref  = best_triple['f1']
    grid4 = grid_4way(step=0.1)
    print(f'4-way simplex grid points: {len(grid4)}')
    for add_model in BASE_REGISTRY:
        if add_model in triple_members: continue
        quad = triple_members + [add_model]
        best = None
        for w in grid4:
            vec = sum(wi * cv_scores[m] for wi, m in zip(w, quad))
            _, f1 = best_f1_on(vec, y_cv)
            if best is None or f1 > best[0]:
                best = (f1, w, vec)
        _, w, vec = best
        rec = score_combo(quad, w, vec, y_cv)
        rec['tag']      = f'4way:{"+".join(quad)}'
        rec['added']    = add_model
        rec['delta_f1'] = round(rec['f1'] - triple_f1_ref, 4)
        quadruple_records.append(rec)

    quads_df = (pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')}
                              for r in quadruple_records])
                .sort_values('f1', ascending=False).reset_index(drop=True))
    show = ['tag','weights','thr','f1','delta_f1','precision','recall',
            'fold_f1_mean','fold_f1_std'] + [f'rec@P{int(t*100)}' for t in PREC_TARGETS]
    print(quads_df[show].to_string(index=False))

    accepted_4way = [r for r in quadruple_records if r['delta_f1'] >= ADD_EPS]
    if accepted_4way:
        best_quadruple = sorted(accepted_4way, key=lambda r: -r['f1'])[0]
        print(f'\nAccepted 4-way winner: {best_quadruple["tag"]}  delta = {best_quadruple["delta_f1"]:+.4f}')
    else:
        print(f'\nNo 4-way beats the 3-way by >= {ADD_EPS:.2f} on CV F1.')
else:
    print('Skipping Phase 3: no accepted 3-way to build on.')


## 10. One-shot test on audios5 (frozen cv_thr -> gap_f1)

Refit every base model on full `audios2+audios4`, score `audios5`, apply frozen cv_thr. This touches
the test set exactly once. Any `gap_f1` > +0.03 flags overfitting-to-train.

In [ ]:
df_tr_all = pd.concat([batches[b] for b in TRAIN_FOLDERS], ignore_index=True)
df_te     = batches[TEST_FOLDER]
y_te      = df_te['label_int'].values

test_scores = {}
for tag, (X_fn, make_clf, _) in BASE_REGISTRY.items():
    test_scores[tag] = fit_and_score(X_fn, make_clf, df_tr_all, df_te)
print(f'Test scored for: {list(test_scores)}')
print(f'Test rows: {len(y_te)}  cheat={int((y_te==1).sum())}  honest={int((y_te==0).sum())}')

# Per-base audios5 reference (NOT used to pick threshold)
base_test_rows = []
for tag in BASE_REGISTRY:
    thr_cv = base_df.loc[base_df['base']==tag, 'thr'].iloc[0]
    cv_f1v = base_df.loc[base_df['base']==tag, 'f1'].iloc[0]
    m  = metrics_at(test_scores[tag], y_te, thr_cv)
    rp = rec_at_prec(test_scores[tag], y_te)
    base_test_rows.append({'base': tag, 'cv_f1': cv_f1v,
                           'test_f1_at_cv':   m['f1'],
                           'test_prec_at_cv': m['precision'],
                           'test_rec_at_cv':  m['recall'],
                           'gap_f1': round(cv_f1v - m['f1'], 4),
                           **{f'test_{k}': v for k, v in rp.items()}})
base_test_df = pd.DataFrame(base_test_rows).sort_values('cv_f1', ascending=False).reset_index(drop=True)
print('\nBase models: CV vs one-shot audios5 at frozen cv_thr:')
print(base_test_df.to_string(index=False))

def freeze_config(rec):
    members = list(rec['members']); weights = list(rec['weights']); thr = rec['thr']
    p_te  = sum(w * test_scores[m] for w, m in zip(weights, members))
    m_te  = metrics_at(p_te, y_te, thr)
    rp_te = rec_at_prec(p_te, y_te)
    cv_rp = {k: rec[k] for k in rec if isinstance(k, str) and k.startswith('rec@P')}
    return {'tag': rec['tag'], 'members': members, 'weights': weights, 'cv_thr': thr,
            'cv_f1': rec['f1'], 'cv_prec': rec['precision'], 'cv_rec': rec['recall'],
            'fold_f1_mean': rec['fold_f1_mean'], 'fold_f1_std': rec['fold_f1_std'],
            'test_f1_at_cv':   m_te['f1'],
            'test_prec_at_cv': m_te['precision'],
            'test_rec_at_cv':  m_te['recall'],
            'gap_f1': round(rec['f1'] - m_te['f1'], 4),
            **{f'cv_{k}': v   for k, v in cv_rp.items()},
            **{f'test_{k}': v for k, v in rp_te.items()}}

finalists = [best_pair]
if best_triple is not None: finalists.append(best_triple)
if best_quadruple is not None: finalists.append(best_quadruple)

final_rows = [freeze_config(r) for r in finalists]
final_df = pd.DataFrame(final_rows)

show = ['tag','members','weights','cv_thr','cv_f1','test_f1_at_cv','gap_f1',
        'test_prec_at_cv','test_rec_at_cv',
        'fold_f1_mean','fold_f1_std'] + \
       [f'cv_rec@P{int(t*100)}' for t in PREC_TARGETS] + \
       [f'test_rec@P{int(t*100)}' for t in PREC_TARGETS]
print('\nPhase winners on audios5 (one-shot):')
print(final_df[[c for c in show if c in final_df.columns]].to_string(index=False))


## 11. Save everything

In [ ]:
import os

base_df.to_csv(SAVE_DIR / 'base_cv_metrics.csv', index=False)
base_folds_df.to_csv(SAVE_DIR / 'base_per_fold.csv', index=False)
pairs_df.to_csv(SAVE_DIR / 'phase1_pairs.csv', index=False)
triples_df.to_csv(SAVE_DIR / 'phase2_triples.csv', index=False)
if isinstance(quads_df, pd.DataFrame) and not quads_df.empty:
    quads_df.to_csv(SAVE_DIR / 'phase3_quads.csv', index=False)
base_test_df.to_csv(SAVE_DIR / 'base_one_shot_test.csv', index=False)
final_df.to_csv(SAVE_DIR / 'phase_winners.csv', index=False)

# Per-file OOF + test probas
oof_out = pd.DataFrame({'filename': fn_cv, 'label_int': y_cv, 'fold': fold_assign})
for m in BASE_REGISTRY: oof_out[f'cv_{m}'] = cv_scores[m]
oof_out.to_csv(SAVE_DIR / 'base_cv_oof.csv', index=False)

test_out = pd.DataFrame({'filename': df_te['filename'].values, 'label_int': y_te})
for m in BASE_REGISTRY: test_out[f'test_{m}'] = test_scores[m]
test_out.to_csv(SAVE_DIR / 'base_test_probas.csv', index=False)

frozen_json = {
    'cv_protocol': {'target': CV_TARGET, 'folds': CV_FOLDS,
                    'deploy_pos_rate': DEPLOY_POS_RATE, 'spw_deploy': round(SPW_DEPLOY, 4)},
    'add_eps': ADD_EPS,
    'top10_feats': TOP10_FEATS,
    'top15_feats': TOP15_FEATS,
    'stylo_feats': stylo_cols,
    'base_cols': {'wavlm': len(wavlm_cols), 'whisper': len(whisper_cols)},
    'phase_winners': final_rows,
}
with open(SAVE_DIR / 'frozen_configs.json', 'w') as f:
    json.dump(frozen_json, f, indent=2, default=str)

print(f'Saved to {SAVE_DIR}/')
for f in sorted(os.listdir(SAVE_DIR)):
    print(f'  {f}')
